In [1]:
# Import library yang dibutuhkan
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.impute import SimpleImputer, KNNImputer
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

print("Library berhasil di-import!")

Library berhasil di-import!


In [2]:
print("\n--- PRAKTIKUM 1: Perbandingan Strategi Imputasi ---")

# 1. Buat dataset buatan
X, y = make_classification(n_samples=1000, n_features=10, n_informative=8, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Buat Missing Value buatan (10% data dihilangkan secara acak)
np.random.seed(42)
mask = np.random.rand(*X_train.shape) < 0.1
X_train_missing = X_train.copy()
X_train_missing[mask] = np.nan

X_test_missing = X_test.copy()
mask_test = np.random.rand(*X_test.shape) < 0.1
X_test_missing[mask_test] = np.nan

# 3. Strategi (a) Mean Imputation
imputer_mean = SimpleImputer(strategy='mean')
X_train_mean = imputer_mean.fit_transform(X_train_missing)
X_test_mean = imputer_mean.transform(X_test_missing)

model_mean = LogisticRegression(max_iter=1000, random_state=42)
model_mean.fit(X_train_mean, y_train)
y_pred_mean = model_mean.predict(X_test_mean)
f1_mean = f1_score(y_test, y_pred_mean)

# 4. Strategi (b) KNN Imputer
imputer_knn = KNNImputer(n_neighbors=5)
X_train_knn = imputer_knn.fit_transform(X_train_missing)
X_test_knn = imputer_knn.transform(X_test_missing)

model_knn = LogisticRegression(max_iter=1000, random_state=42)
model_knn.fit(X_train_knn, y_train)
y_pred_knn = model_knn.predict(X_test_knn)
f1_knn = f1_score(y_test, y_pred_knn)

# 5. Tampilkan Hasil
print(f"F1-Score (Mean Imputation) : {f1_mean:.4f}")
print(f"F1-Score (KNN Imputer)     : {f1_knn:.4f}")

if f1_mean > f1_knn:
    print("Kesimpulan: Mean Imputation lebih baik pada kasus ini.")
else:
    print("Kesimpulan: KNN Imputer lebih baik pada kasus ini.")


--- PRAKTIKUM 1: Perbandingan Strategi Imputasi ---
F1-Score (Mean Imputation) : 0.6599
F1-Score (KNN Imputer)     : 0.6832
Kesimpulan: KNN Imputer lebih baik pada kasus ini.


In [3]:
print("\n--- PRAKTIKUM 2: Efektivitas Penanganan Imbalanced Data ---")

# 1. Buat dataset IMBALANCED (misal 95% kelas 0, 5% kelas 1)
X_imb, y_imb = make_classification(n_samples=2000, n_features=10, weights=[0.95], random_state=42)
X_train_imb, X_test_imb, y_train_imb, y_test_imb = train_test_split(X_imb, y_imb, test_size=0.2, random_state=42)

# --- (A) BASELINE (Tanpa Penanganan) ---
model_base = LogisticRegression(max_iter=1000, random_state=42)
model_base.fit(X_train_imb, y_train_imb)
y_pred_base = model_base.predict(X_test_imb)
f1_base = f1_score(y_test_imb, y_pred_base, pos_label=1)

# --- (B) SMOTE (Oversampling) ---
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_imb, y_train_imb)
model_smote = LogisticRegression(max_iter=1000, random_state=42)
model_smote.fit(X_train_smote, y_train_smote)
y_pred_smote = model_smote.predict(X_test_imb)
f1_smote = f1_score(y_test_imb, y_pred_smote, pos_label=1)

# --- (C) UNDERSAMPLING (Sesuai gambar modul) ---
rus = RandomUnderSampler(random_state=42)
X_train_rus, y_train_rus = rus.fit_resample(X_train_imb, y_train_imb)
model_rus = LogisticRegression(max_iter=1000, random_state=42)
model_rus.fit(X_train_rus, y_train_rus)
y_pred_rus = model_rus.predict(X_test_imb)
f1_rus = f1_score(y_test_imb, y_pred_rus, pos_label=1)

# --- (D) CLASS WEIGHT (Sesuai gambar modul) ---
model_cw = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
model_cw.fit(X_train_imb, y_train_imb)
y_pred_cw = model_cw.predict(X_test_imb)
f1_cw = f1_score(y_test_imb, y_pred_cw, pos_label=1)

# Tampilkan Hasil Perbandingan
print(f"F1-Score Baseline       : {f1_base:.4f}")
print(f"F1-Score SMOTE          : {f1_smote:.4f}")
print(f"F1-Score Undersampling  : {f1_rus:.4f}")
print(f"F1-Score Class Weight   : {f1_cw:.4f}")

# Kesimpulan
scores = {'Baseline': f1_base, 'SMOTE': f1_smote, 'Undersampling': f1_rus, 'Class Weight': f1_cw}
best_method = max(scores, key=scores.get)
print(f"\nKesimpulan: Pendekatan yang paling efektif untuk kasus ini adalah **{best_method}** dengan F1-Score tertinggi ({scores[best_method]:.4f}).")


--- PRAKTIKUM 2: Efektivitas Penanganan Imbalanced Data ---
F1-Score Baseline       : 0.6818
F1-Score SMOTE          : 0.5854
F1-Score Undersampling  : 0.5455
F1-Score Class Weight   : 0.5647

Kesimpulan: Pendekatan yang paling efektif untuk kasus ini adalah **Baseline** dengan F1-Score tertinggi (0.6818).
